In [ ]:
# parallel workflow
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [14]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str



In [15]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls'])/100
    return {'sr': sr}

def calculate_bpb(state: BatsmanState):
    bpb = state['balls']/(state['fours'] + state['sixes'])
    return {'bpb': bpb}

def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = ((state['fours']*4) + (state['sixes']*6))/state['runs']
    return {'boundary_percent': boundary_percent}


In [16]:
def summary(state: BatsmanState):
    summary = f"""
        Strike Rate - {state['sr']} \n
        Balls per boundary - {state['bpb']} \n
        Boundary percent - {state['boundary_percent']}
    """

    return {'summary': summary}

In [17]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()


In [18]:
inital_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(inital_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 0.02,
 'bpb': 5.0,
 'boundary_percent': 0.48,
 'summary': '\n        Strike Rate - 0.02 \n\n        Balls per boundary - 5.0 \n\n        Boundary percent - 0.48\n    '}